# Notebook 1: Fetch NBA Data
Pulls 2024-25 NBA player box score data using `nba_api` and saves it as CSV files into `data/`.

**Run this once.** Step 3 takes 20–40 minutes due to rate limiting.

In [ ]:
import pandas as pd
import time
import os
import random
from nba_api.stats.endpoints import leaguegamelog, boxscoretraditionalv3
from nba_api.stats.static import players, teams

os.makedirs('data', exist_ok=True)
print('Libraries loaded.')

## Step 1: Fetch All Games for 2024-25 Season

In [ ]:
season_types = [
    ('Regular Season', '02'),
    ('Playoffs', '04'),
]

all_games_list = []

for season_label, season_id_prefix in season_types:
    log = leaguegamelog.LeagueGameLog(
        season='2024-25',
        season_type_all_star=season_label,
        timeout=60
    )
    df = log.get_data_frames()[0]
    df['SEASON_TYPE'] = season_label
    all_games_list.append(df)
    print(f'{season_label}: {len(df)} team-game records')

all_games = pd.concat(all_games_list, ignore_index=True)
print(f'Total team-game records: {len(all_games)}')
all_games.head()

## Step 2: Build Games Table

In [ ]:
# Deduplicate to one row per game
games_df = all_games[['GAME_ID', 'GAME_DATE', 'MATCHUP', 'SEASON_TYPE']].drop_duplicates(subset='GAME_ID').copy()

# Parse home/away from MATCHUP (e.g. 'BOS vs. NYK' or 'NYK @ BOS')
def parse_matchup(row):
    if 'vs.' in row['MATCHUP']:
        home = row['MATCHUP'].split(' vs. ')[0].strip()
        away = row['MATCHUP'].split(' vs. ')[1].strip()
    else:
        away = row['MATCHUP'].split(' @ ')[0].strip()
        home = row['MATCHUP'].split(' @ ')[1].strip()
    return home, away

games_df[['HOME_TEAM', 'AWAY_TEAM']] = games_df.apply(
    lambda r: pd.Series(parse_matchup(r)), axis=1
)

games_df = games_df[['GAME_ID', 'GAME_DATE', 'HOME_TEAM', 'AWAY_TEAM', 'SEASON_TYPE']]
games_df.to_csv('data/games.csv', index=False)
print(f'Games saved: {len(games_df)}')
games_df.head()

## Step 3: Fetch Player Box Scores
> ⚠️ This step makes one API call per game with a short delay to avoid rate limiting. It may take **20–40 minutes** for the full season.

In [ ]:
game_ids = games_df['GAME_ID'].unique().tolist()
all_boxscores = []
failed = []

for i, gid in enumerate(game_ids):
    try:
        bs = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=gid, timeout=60)
        df = bs.get_data_frames()[0]
        all_boxscores.append(df)
        if (i + 1) % 50 == 0:
            print(f'Fetched {i+1}/{len(game_ids)} games...')
        time.sleep(random.randint(1, 5))
    except Exception as e:
        print(f'Failed game {gid}: {e}')
        failed.append(gid)
        time.sleep(random.randint(1, 5))

boxscores_df = pd.concat(all_boxscores, ignore_index=True)
print(f'Total player-game records: {len(boxscores_df)}')
if failed:
    print(f'Failed game IDs: {failed}')

## Step 4: Build PlayerBoxScores and Players Tables

In [ ]:
# Select relevant columns
box_cols = [
    'GAME_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION',
    'MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV',
    'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT', 'PLUS_MINUS'
]
player_boxscores = boxscores_df[box_cols].copy()

# Merge in game date and season type from Games table
player_boxscores = player_boxscores.merge(
    games_df[['GAME_ID', 'GAME_DATE', 'SEASON_TYPE']],
    on='GAME_ID', how='left'
)

player_boxscores.to_csv('data/player_boxscores.csv', index=False)
print(f'PlayerBoxScores saved: {len(player_boxscores)} rows')

# Build Players table (most recent team per player)
players_df = (
    player_boxscores
    .sort_values('GAME_DATE')
    .drop_duplicates(subset='PLAYER_ID', keep='last')
    [['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION']]
    .rename(columns={'TEAM_ABBREVIATION': 'TEAM'})
)
players_df.to_csv('data/players.csv', index=False)
print(f'Players saved: {len(players_df)} unique players')
player_boxscores.head()